# Notebook 05: Feature Engineering

## Step 1: Environment Setup & Spark Session

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("SmartCityBusClustering_FeatureEng") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/24 09:46:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/24 09:46:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/24 09:46:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/07/24 09:46:46 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark version: 4.1.1


## Step 2: Load Cleaned Dataset

In [2]:
PROJECT_ROOT = "/Users/aayushbohara/Desktop/smartcity-bus-clustering"
CLEANED_PATH = os.path.join(PROJECT_ROOT, "data/processed/cleaned_dataset.csv")
FEATURES_OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data/processed/features_dataset.csv")

df = spark.read.csv(CLEANED_PATH, header=True, inferSchema=True)
print("Row count:", df.count())
df.printSchema()

Row count: 771733
root
 |-- timestamp: timestamp (nullable = true)
 |-- lineRef: string (nullable = true)
 |-- directionRef: string (nullable = true)
 |-- vehicleRef: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location_source_file: string (nullable = true)
 |-- lineName: string (nullable = true)
 |-- serviceCode: string (nullable = true)
 |-- operator: string (nullable = true)
 |-- nationalOperatorCode: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- vehicleJourneyCode: string (nullable = true)
 |-- departureTime: timestamp (nullable = true)
 |-- journeyPatternRef: string (nullable = true)
 |-- timetable_source_file: string (nullable = true)
 |-- fare_organisation: string (nullable = true)
 |-- common_product_type: string (nullable = true)
 |-- common_tariff_basis: string (nullable = true)
 |-- common_product_name: string (nullable = true)
 |-- fare_product

## Step 3: Temporal Features

In [3]:
from pyspark.sql.functions import hour, dayofweek, when

df = df.withColumn("hour_of_day", hour("timestamp"))

# peak/off-peak bucket (typical UK commute peaks: 7-9am, 4-6pm)
# NOTE: given the EDA finding that our collection window is biased toward
# evening hours, this feature carries the same documented limitation.
df = df.withColumn(
    "day_period",
    when((col("hour_of_day") >= 7) & (col("hour_of_day") <= 9), "morning_peak")
    .when((col("hour_of_day") >= 16) & (col("hour_of_day") <= 18), "evening_peak")
    .when((col("hour_of_day") >= 22) | (col("hour_of_day") <= 5), "night")
    .otherwise("off_peak")
)

df.groupBy("day_period").count().show()

+------------+------+
|  day_period| count|
+------------+------+
|    off_peak|558717|
|       night|192402|
|morning_peak|   570|
|evening_peak| 20044|
+------------+------+



## Step 4: Schedule Deviation Feature

In [6]:
from pyspark.sql.functions import date_format

# extract time-of-day as minutes-since-midnight for both timestamp and departureTime
df = df.withColumn("ping_time_of_day_min",
    (hour("timestamp") * 60 + minute("timestamp"))
) if False else df  # placeholder removed below

from pyspark.sql.functions import minute

df = df.withColumn("ping_minutes_since_midnight", hour("timestamp") * 60 + minute("timestamp"))
df = df.withColumn("scheduled_minutes_since_midnight",
    when(col("has_timetable_match"), hour("departureTime") * 60 + minute("departureTime")).otherwise(None)
)

# raw difference, then handle day-wraparound (e.g. ping at 00:10, scheduled at 23:50 -> should be +20, not -1420)
raw_diff = col("ping_minutes_since_midnight") - col("scheduled_minutes_since_midnight")
df = df.withColumn(
    "schedule_deviation_minutes",
    when(raw_diff > 720, raw_diff - 1440)
    .when(raw_diff < -720, raw_diff + 1440)
    .otherwise(raw_diff)
)

df.select("schedule_deviation_minutes").describe().show()

+-------+--------------------------+
|summary|schedule_deviation_minutes|
+-------+--------------------------+
|  count|                    769310|
|   mean|        -342.9699600941103|
| stddev|        298.27756277720255|
|    min|                      -719|
|    max|                       720|
+-------+--------------------------+



## Step 5: Vehicle Speed Estimate

In [7]:
# distinct real GPS pings only (dedup the join fan-out before computing speed)
distinct_pings = df.select("timestamp", "vehicleRef", "latitude", "longitude").distinct()

w = Window.partitionBy("vehicleRef").orderBy("timestamp")

distinct_pings = distinct_pings \
    .withColumn("prev_lat", lag("latitude").over(w)) \
    .withColumn("prev_lon", lag("longitude").over(w)) \
    .withColumn("prev_timestamp", lag("timestamp").over(w))

R = 6371.0
distinct_pings = distinct_pings.withColumn(
    "distance_km",
    when(col("prev_lat").isNotNull(),
        2 * R * atan2(
            sqrt(sin(radians(col("latitude") - col("prev_lat")) / 2) ** 2 +
                 cos(radians(col("prev_lat"))) * cos(radians(col("latitude"))) *
                 sin(radians(col("longitude") - col("prev_lon")) / 2) ** 2),
            sqrt(1 - (sin(radians(col("latitude") - col("prev_lat")) / 2) ** 2 +
                      cos(radians(col("prev_lat"))) * cos(radians(col("latitude"))) *
                      sin(radians(col("longitude") - col("prev_lon")) / 2) ** 2))
        )
    ).otherwise(None)
)

distinct_pings = distinct_pings.withColumn(
    "time_diff_minutes",
    when(col("prev_timestamp").isNotNull(),
         (unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")) / 60.0
    ).otherwise(None)
)

distinct_pings = distinct_pings.withColumn(
    "speed_kmh",
    when(col("time_diff_minutes") > 0, col("distance_km") / (col("time_diff_minutes") / 60.0)).otherwise(None)
)

# join speed features back onto the full (duplicated) table
speed_lookup = distinct_pings.select("timestamp", "vehicleRef", "distance_km", "time_diff_minutes", "speed_kmh")

df = df.drop("distance_km", "time_diff_minutes", "speed_kmh", "prev_lat", "prev_lon", "prev_timestamp") \
       .join(speed_lookup, on=["timestamp", "vehicleRef"], how="left")

df.select("distance_km", "time_diff_minutes", "speed_kmh").describe().show()

+-------+------------------+--------------------+------------------+
|summary|       distance_km|   time_diff_minutes|         speed_kmh|
+-------+------------------+--------------------+------------------+
|  count|            661763|              661763|            661763|
|   mean|0.7817602442372885|     9.1453520369081|12.109155361774407|
| stddev|1.8441608581437572|   37.53600595034486|142.98996719080702|
|    min|               0.0|0.016666666666666666|               0.0|
|    max| 34.02026924570951|              1709.6| 36717.74712402493|
+-------+------------------+--------------------+------------------+



## Step 6: Fix Schedule Deviation — Nearest Match Only

In [8]:
from pyspark.sql.functions import min as spark_min, abs as spark_abs

nearest_deviation = df.filter(col("schedule_deviation_minutes").isNotNull()) \
    .withColumn("abs_deviation", spark_abs(col("schedule_deviation_minutes"))) \
    .groupBy("timestamp", "vehicleRef") \
    .agg(spark_min("abs_deviation").alias("nearest_schedule_deviation_minutes"))

df = df.drop("schedule_deviation_minutes") \
       .join(nearest_deviation, on=["timestamp", "vehicleRef"], how="left")

df.select("nearest_schedule_deviation_minutes").describe().show()

+-------+----------------------------------+
|summary|nearest_schedule_deviation_minutes|
+-------+----------------------------------+
|  count|                            769310|
|   mean|                312.88070998687135|
| stddev|                172.10146275271953|
|    min|                                 0|
|    max|                               687|
+-------+----------------------------------+



## Step 7: Cap Unrealistic Speed Values

In [9]:
SPEED_CAP_KMH = 120

outlier_count = df.filter(col("speed_kmh") > SPEED_CAP_KMH).count()
print(f"Rows with speed > {SPEED_CAP_KMH} km/h (treated as outliers): {outlier_count}")

df = df.withColumn(
    "speed_kmh_capped",
    when(col("speed_kmh") > SPEED_CAP_KMH, None).otherwise(col("speed_kmh"))
)

df.select("speed_kmh_capped").describe().show()

Rows with speed > 120 km/h (treated as outliers): 90
+-------+------------------+
|summary|  speed_kmh_capped|
+-------+------------------+
|  count|            661673|
|   mean|11.516991958878759|
| stddev| 8.646611219421818|
|    min|               0.0|
|    max| 86.64601649248249|
+-------+------------------+



## Step 8 (Revised): Re-parse Timetable with Time-Spread Sampling

In [10]:
import glob
import xml.etree.ElementTree as ET

PROJECT_ROOT = "/Users/aayushbohara/Desktop/smartcity-bus-clustering"
timetable_files = glob.glob(os.path.join(PROJECT_ROOT, "data/raw/Timestabledata", "**", "*.xml"), recursive=True)
print("Timetable files found:", len(timetable_files))

timetable_files_rdd = spark.sparkContext.parallelize(timetable_files, numSlices=8)

def parse_all_departures(iterator):
    rows = []
    for path in iterator:
        try:
            tree = ET.parse(path)
            root = tree.getroot()
        except ET.ParseError:
            continue

        def find(elem, tag):
            for e in elem.iter():
                if e.tag.endswith(tag):
                    return e.text
            return None

        line_name = find(root, "LineName")
        if line_name is None:
            continue

        for vj in root.iter():
            if vj.tag.endswith("VehicleJourney"):
                departure = find(vj, "DepartureTime")
                if departure:
                    rows.append((line_name, departure))
    return rows

all_departures_rdd = timetable_files_rdd.mapPartitions(parse_all_departures)
print("Total raw departure entries parsed:", all_departures_rdd.count())

Timetable files found: 1318


[Stage 68:==================================================>       (7 + 1) / 8]

Total raw departure entries parsed: 41275


## Step 9: Build Time-Spread Timetable Sample & Convert to Minutes-Since-Midnight

In [11]:
from pyspark.sql.types import StructType, StructField, StringType

departure_schema = StructType([
    StructField("lineName", StringType(), True),
    StructField("departureTimeStr", StringType(), True),
])

all_departures_df = spark.createDataFrame(all_departures_rdd, schema=departure_schema)

# parse "HH:MM:SS" into minutes-since-midnight directly (avoids the earlier date-attachment bug entirely)
from pyspark.sql.functions import split, col as col2

all_departures_df = all_departures_df.withColumn("time_parts", split(col2("departureTimeStr"), ":"))
all_departures_df = all_departures_df.withColumn("dep_hour", col2("time_parts")[0].cast("int"))
all_departures_df = all_departures_df.withColumn("dep_minute", col2("time_parts")[1].cast("int"))
all_departures_df = all_departures_df.withColumn("dep_minutes_since_midnight", col2("dep_hour") * 60 + col2("dep_minute"))

all_departures_df = all_departures_df.select("lineName", "dep_minutes_since_midnight").dropna()

print("Parsed departure rows:", all_departures_df.count())
all_departures_df.show(5)

Parsed departure rows: 41275


[Stage 72:>                                                         (0 + 1) / 1]

+--------+--------------------------+
|lineName|dep_minutes_since_midnight|
+--------+--------------------------+
|     875|                       910|
|     875|                       455|
|      57|                       912|
|     348|                       425|
|     348|                       455|
+--------+--------------------------+
only showing top 5 rows


## Step 10: Sample 10 Time-Spread Journeys Per Line


In [13]:
from pyspark.sql.functions import row_number

w2 = Window.partitionBy("lineName", "time_bucket").orderBy("dep_minutes_since_midnight")
timetable_spread = all_departures_df.withColumn("rn", row_number().over(w2)) \
    .filter(col2("rn") == 1) \
    .select("lineName", "dep_minutes_since_midnight")

print("Time-spread sampled timetable rows:", timetable_spread.count())
timetable_spread.groupBy("lineName").count().orderBy(col2("count").desc()).show(10)

Time-spread sampled timetable rows: 2344


[Stage 79:===========================================>              (6 + 2) / 8]

+--------+-----+
|lineName|count|
+--------+-----+
|     521|   10|
|     142|   10|
|     101|   10|
|     201|   10|
|     103|   10|
|     409|   10|
|     118|   10|
|     471|   10|
|      50|   10|
|      36|   10|
+--------+-----+
only showing top 10 rows


## Step 11: Recompute Schedule Deviation Using Time-Spread Sample

In [14]:
# join full dataset against the corrected time-spread timetable sample on lineName
df_joined = df.withColumn("ping_minutes_since_midnight_v2", hour("timestamp") * 60 + minute("timestamp")) \
    .join(
        timetable_spread.withColumnRenamed("lineName", "lineName_spread"),
        df["lineName"] == col2("lineName_spread"),
        how="left"
    )

# compute deviation against every matched sampled journey, handling day-wraparound
raw_diff_v2 = col2("ping_minutes_since_midnight_v2") - col2("dep_minutes_since_midnight")

df_joined = df_joined.withColumn(
    "deviation_v2",
    when(raw_diff_v2 > 720, raw_diff_v2 - 1440)
    .when(raw_diff_v2 < -720, raw_diff_v2 + 1440)
    .otherwise(raw_diff_v2)
)

# take the minimum absolute deviation per real ping (nearest scheduled journey)
nearest_deviation_v2 = df_joined.filter(col2("deviation_v2").isNotNull()) \
    .withColumn("abs_deviation_v2", spark_abs(col2("deviation_v2"))) \
    .groupBy("timestamp", "vehicleRef") \
    .agg(spark_min("abs_deviation_v2").alias("nearest_schedule_deviation_minutes_v2"))

df = df.drop("nearest_schedule_deviation_minutes") \
       .join(nearest_deviation_v2, on=["timestamp", "vehicleRef"], how="left")

df.select("nearest_schedule_deviation_minutes_v2").describe().show()

+-------+-------------------------------------+
|summary|nearest_schedule_deviation_minutes_v2|
+-------+-------------------------------------+
|  count|                               769310|
|   mean|                    51.79197462661346|
| stddev|                   60.621451756888746|
|    min|                                    0|
|    max|                                  466|
+-------+-------------------------------------+



## Step 12: Assemble Final Feature Vector (VectorAssembler + StandardScaler)

In [15]:
from pyspark.sql.functions import coalesce, lit

# fill nulls with a neutral value before vectorizing (StandardScaler can't handle nulls)
df_model_ready = df.withColumn(
    "nearest_schedule_deviation_minutes_v2",
    coalesce(col("nearest_schedule_deviation_minutes_v2"), lit(-1))  # -1 flags "no timetable match"
).withColumn(
    "speed_kmh_capped",
    coalesce(col("speed_kmh_capped"), lit(-1))  # -1 flags "no prior ping to compute speed"
)

feature_cols = [
    "latitude", "longitude", "hour_of_day",
    "nearest_schedule_deviation_minutes_v2", "speed_kmh_capped", "disruption_count"
]

from pyspark.sql.functions import sum as spark_sum, when as when2
null_check = df_model_ready.select([
    spark_sum(when2(col(c).isNull(), 1).otherwise(0)).alias(c) for c in feature_cols
])
print("Final null check before vectorizing:")
null_check.show()

Final null check before vectorizing:
+--------+---------+-----------+-------------------------------------+----------------+----------------+
|latitude|longitude|hour_of_day|nearest_schedule_deviation_minutes_v2|speed_kmh_capped|disruption_count|
+--------+---------+-----------+-------------------------------------+----------------+----------------+
|       0|        0|          0|                                    0|               0|               0|
+--------+---------+-----------+-------------------------------------+----------------+----------------+



## Step 13: VectorAssembler + StandardScaler Pipeline

In [16]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

feature_pipeline = Pipeline(stages=[assembler, scaler])
feature_pipeline_model = feature_pipeline.fit(df_model_ready)
df_features = feature_pipeline_model.transform(df_model_ready)

df_features.select("features_raw", "features").show(5, truncate=False)

+-------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------+
|features_raw                                           |features                                                                                                               |
+-------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------+
|[53.513843,-2.195335,20.0,44.0,17.216607377044912,52.0]|[0.39752460773144527,0.5366941121013674,0.12354557856738234,-0.1258489704593358,0.8202753151651043,0.13980180433832742]|
|[53.513843,-2.195335,20.0,44.0,17.216607377044912,52.0]|[0.39752460773144527,0.5366941121013674,0.12354557856738234,-0.1258489704593358,0.8202753151651043,0.13980180433832742]|
|[53.513843,-2.195335,20.0,44.0,17.216607377044912,52.0]|[0.39752460773144527,0.5366941121013674,0.12354557856

## Step 14: Export Feature-Engineered Dataset

In [17]:
FEATURES_PARQUET_PATH = os.path.join(PROJECT_ROOT, "data/processed/features_dataset.parquet")

df_features.select(
    "timestamp", "lineRef", "vehicleRef", "nationalOperatorCode",
    "latitude", "longitude", "hour_of_day", "day_period",
    "nearest_schedule_deviation_minutes_v2", "speed_kmh_capped", "disruption_count",
    "features_raw", "features"
).write.mode("overwrite").parquet(FEATURES_PARQUET_PATH)

print("Feature-engineered dataset exported to:", FEATURES_PARQUET_PATH)

# verify
verify = spark.read.parquet(FEATURES_PARQUET_PATH)
print("Row count:", verify.count())
verify.printSchema()

Feature-engineered dataset exported to: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/features_dataset.parquet
Row count: 771733
root
 |-- timestamp: timestamp (nullable = true)
 |-- lineRef: string (nullable = true)
 |-- vehicleRef: string (nullable = true)
 |-- nationalOperatorCode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_period: string (nullable = true)
 |-- nearest_schedule_deviation_minutes_v2: integer (nullable = true)
 |-- speed_kmh_capped: double (nullable = true)
 |-- disruption_count: integer (nullable = true)
 |-- features_raw: vector (nullable = true)
 |-- features: vector (nullable = true)

